# Scaled Dot-Product Attention do Zero

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Atenção permite que cada token olhe para todos os outros e agregue informação ponderada pela *relevância*. O query faz uma pergunta, as keys anunciam conteúdo, os values são o que de fato é somado. A divisão por $\sqrt{d_k}$ mantém os produtos internos pequenos para a softmax não saturar.


## Formulação Matemática

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

com $Q \in \mathbb{R}^{n \times d_k}$, $K \in \mathbb{R}^{m \times d_k}$, $V \in \mathbb{R}^{m \times d_v}$.

- Cada linha de $QK^\top$ é a similaridade não normalizada entre uma query e todas as keys.
- A softmax sobre a última dimensão transforma esses scores em uma distribuição de probabilidade.
- Multiplicar por $V$ produz uma média ponderada dos vetores de value.


## Implementação


In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(0)


In [ ]:
def attention_numpy(Q, K, V, mask=None):
    """Reference implementation in pure NumPy."""
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    return weights @ V, weights


In [ ]:
def attention_torch(Q, K, V, mask=None):
    """Same thing with autograd-friendly tensors."""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


## Experimento


In [ ]:
n, m, d_k, d_v = 4, 6, 8, 8
Q = torch.randn(n, d_k)
K = torch.randn(m, d_k)
V = torch.randn(m, d_v)

out, w = attention_torch(Q, K, V)
print('out shape:', out.shape)
print('attn weights row sums:', w.sum(-1))


In [ ]:
# sanity-check against torch's built-in scaled_dot_product_attention
ref = F.scaled_dot_product_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)).squeeze(0)
print('max diff:', (out - ref).abs().max().item())


## Discussão

- O fator $\sqrt{d_k}$ importa: sem ele a softmax satura quando $d_k$ é grande, gerando gradientes minúsculos.
- A máscara é o que diferencia atenção causal (decoder) de bidirecional (encoder).
- Para sequências longas, o custo de memória $O(n^2)$ é o gargalo — ver *FlashAttention* e *Linear Attention* como soluções.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
